In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/dlp-26t2-nppe3/sample_submission.csv
/kaggle/input/competitions/dlp-26t2-nppe3/submission.py
/kaggle/input/competitions/dlp-26t2-nppe3/archive/val/gt/gt_00159.png
/kaggle/input/competitions/dlp-26t2-nppe3/archive/val/gt/gt_00056.png
/kaggle/input/competitions/dlp-26t2-nppe3/archive/val/gt/gt_00017.png
/kaggle/input/competitions/dlp-26t2-nppe3/archive/val/gt/gt_00124.png
/kaggle/input/competitions/dlp-26t2-nppe3/archive/val/gt/gt_00140.png
/kaggle/input/competitions/dlp-26t2-nppe3/archive/val/gt/gt_00068.png
/kaggle/input/competitions/dlp-26t2-nppe3/archive/val/gt/gt_00019.png
/kaggle/input/competitions/dlp-26t2-nppe3/archive/val/gt/gt_00266.png
/kaggle/input/competitions/dlp-26t2-nppe3/archive/val/gt/gt_00236.png
/kaggle/input/competitions/dlp-26t2-nppe3/archive/val/gt/gt_00148.png
/kaggle/input/competitions/dlp-26t2-nppe3/archive/val/gt/gt_00152.png
/kaggle/input/competitions/dlp-26t2-nppe3/archive/val/gt/gt_00226.png
/kaggle/input/competitions/dlp-26t2-nppe

In [2]:
import os
import glob
import time
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
import math

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.device_count() > 1:
    print(f'Multiple GPUs found: {torch.cuda.device_count()} GPUs. We will use DataParallel.')

Using device: cuda
Multiple GPUs found: 2 GPUs. We will use DataParallel.


In [3]:
class SRDataset(Dataset):
    def __init__(self, input_dir, gt_dir=None, patch_size=None):
        self.input_paths = sorted(glob.glob(os.path.join(input_dir, '*.png')))
        # In some directory structures, inputs and GTs might have different prefixes but same order,
        # or they might need matching by ID. Sorting them usually works if names correspond directly.
        self.gt_paths = sorted(glob.glob(os.path.join(gt_dir, '*.png'))) if gt_dir else None
        self.patch_size = patch_size # Size for LR patch (HR patch will be 4x)

    def __len__(self):
        return len(self.input_paths)

    def __getitem__(self, idx):
        input_img = Image.open(self.input_paths[idx]).convert('RGB')
        
        if self.gt_paths:
            gt_img = Image.open(self.gt_paths[idx]).convert('RGB')
            
            if self.patch_size:
                w, h = input_img.size
                th, tw = self.patch_size, self.patch_size
                if w >= tw and h >= th:
                    i = torch.randint(0, h - th + 1, size=(1, )).item()
                    j = torch.randint(0, w - tw + 1, size=(1, )).item()
                    
                    input_img = TF.crop(input_img, i, j, th, tw)
                    gt_img = TF.crop(gt_img, i * 4, j * 4, th * 4, tw * 4)
            
            # Data Augmentation
            if torch.rand(1) > 0.5:
                input_img = TF.hflip(input_img)
                gt_img = TF.hflip(gt_img)
            if torch.rand(1) > 0.5:
                input_img = TF.vflip(input_img)
                gt_img = TF.vflip(gt_img)
                
            return TF.to_tensor(input_img), TF.to_tensor(gt_img)
        
        # For test set
        return TF.to_tensor(input_img), os.path.basename(self.input_paths[idx])

# Data Paths
DATA_ROOT = '/kaggle/input/competitions/dlp-26t2-nppe3/archive'
TRAIN_LR_DIR = f'{DATA_ROOT}/train/train'
TRAIN_HR_DIR = f'{DATA_ROOT}/train/gt'
VAL_LR_DIR = f'{DATA_ROOT}/val/val'
VAL_HR_DIR = f'{DATA_ROOT}/val/gt'
TEST_LR_DIR = f'{DATA_ROOT}/test'

BATCH_SIZE = 16
PATCH_SIZE = 64 # Input patch size (Output will be 256x256)

# Initialize Datasets
train_dataset = SRDataset(TRAIN_LR_DIR, TRAIN_HR_DIR, patch_size=PATCH_SIZE)
val_dataset = SRDataset(VAL_LR_DIR, VAL_HR_DIR, patch_size=None) # No cropping for validation
test_dataset = SRDataset(TEST_LR_DIR, gt_dir=None, patch_size=None)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}, Test samples: {len(test_dataset)}')

Train samples: 1105, Val samples: 267, Test samples: 60


In [5]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x):
        return x + self.conv2(self.relu(self.conv1(x)))

class SuperResolutionNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, num_res_blocks=16, num_features=64):
        super(SuperResolutionNet, self).__init__()
        
        # Initial feature extraction
        self.conv_initial = nn.Conv2d(in_channels, num_features, kernel_size=3, padding=1)
        
        # Residual blocks
        self.res_blocks = nn.Sequential(*[ResidualBlock(num_features) for _ in range(num_res_blocks)])
        self.conv_mid = nn.Conv2d(num_features, num_features, kernel_size=3, padding=1)
        
        # Upsampling (4x requires two PixelShuffle layers with 2x each)
        self.upsample1 = nn.Conv2d(num_features, num_features * 4, kernel_size=3, padding=1)
        self.pixel_shuffle1 = nn.PixelShuffle(2)
        self.relu1 = nn.ReLU(inplace=True)
        
        self.upsample2 = nn.Conv2d(num_features, num_features * 4, kernel_size=3, padding=1)
        self.pixel_shuffle2 = nn.PixelShuffle(2)
        self.relu2 = nn.ReLU(inplace=True)
        
        # Final output layer
        self.conv_final = nn.Conv2d(num_features, out_channels, kernel_size=3, padding=1)

    def forward(self, x):
        x = self.conv_initial(x)
        res = self.res_blocks(x)
        res = self.conv_mid(res)
        x = x + res
        
        x = self.upsample1(x)
        x = self.pixel_shuffle1(x)
        x = self.relu1(x)
        
        x = self.upsample2(x)
        x = self.pixel_shuffle2(x)
        x = self.relu2(x)
        
        x = self.conv_final(x)
        return torch.sigmoid(x) # Constrain output to [0, 1] for images

model = SuperResolutionNet(num_res_blocks=16, num_features=64)

# Use multiple GPUs if available (T4x2)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

model = model.to(device)

In [6]:
criterion = nn.L1Loss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

def calc_psnr(img1, img2):
    mse = torch.mean((img1 - img2) ** 2)
    if mse == 0:
        return 100
    return 20 * math.log10(1.0 / math.sqrt(mse))

EPOCHS = 50
best_psnr = 0.0

In [7]:
for epoch in range(EPOCHS):
    # --- Training ---
    model.train()
    train_loss = 0.0
    
    for batch_idx, (lr_imgs, hr_imgs) in enumerate(train_loader):
        lr_imgs, hr_imgs = lr_imgs.to(device), hr_imgs.to(device)
        
        optimizer.zero_grad()
        sr_imgs = model(lr_imgs)
        loss = criterion(sr_imgs, hr_imgs)
        
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
        if batch_idx % 100 == 0:
            print(f'Epoch [{epoch+1}/{EPOCHS}], Step [{batch_idx}/{len(train_loader)}], Loss: {loss.item():.4f}')
            
    train_loss /= len(train_loader)
    scheduler.step()
    
    # --- Validation ---
    model.eval()
    val_psnr = 0.0
    val_loss = 0.0
    with torch.no_grad():
        for lr_imgs, hr_imgs in val_loader:
            lr_imgs, hr_imgs = lr_imgs.to(device), hr_imgs.to(device)
            sr_imgs = model(lr_imgs)
            
            loss = criterion(sr_imgs, hr_imgs)
            val_loss += loss.item()
            
            # Calculate PSNR
            val_psnr += calc_psnr(sr_imgs, hr_imgs)
            
    val_loss /= len(val_loader)
    val_psnr /= len(val_loader)
    
    print(f'==> Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val PSNR: {val_psnr:.2f} dB')
    
    # Save best model
    if val_psnr > best_psnr:
        best_psnr = val_psnr
        torch.save(model.state_dict(), 'best_model.pth')
        print(f'Saved new best model with PSNR: {best_psnr:.2f} dB')

Epoch [1/50], Step [0/70], Loss: 0.3058
==> Epoch [1/50] | Train Loss: 0.0869 | Val Loss: 0.0183 | Val PSNR: 30.33 dB
Saved new best model with PSNR: 30.33 dB
Epoch [2/50], Step [0/70], Loss: 0.0247
==> Epoch [2/50] | Train Loss: 0.0219 | Val Loss: 0.0133 | Val PSNR: 33.63 dB
Saved new best model with PSNR: 33.63 dB
Epoch [3/50], Step [0/70], Loss: 0.0190
==> Epoch [3/50] | Train Loss: 0.0179 | Val Loss: 0.0122 | Val PSNR: 35.01 dB
Saved new best model with PSNR: 35.01 dB
Epoch [4/50], Step [0/70], Loss: 0.0166
==> Epoch [4/50] | Train Loss: 0.0157 | Val Loss: 0.0113 | Val PSNR: 35.70 dB
Saved new best model with PSNR: 35.70 dB
Epoch [5/50], Step [0/70], Loss: 0.0150
==> Epoch [5/50] | Train Loss: 0.0150 | Val Loss: 0.0165 | Val PSNR: 33.55 dB
Epoch [6/50], Step [0/70], Loss: 0.0219
==> Epoch [6/50] | Train Loss: 0.0147 | Val Loss: 0.0120 | Val PSNR: 35.51 dB
Epoch [7/50], Step [0/70], Loss: 0.0156
==> Epoch [7/50] | Train Loss: 0.0136 | Val Loss: 0.0107 | Val PSNR: 36.38 dB
Saved new 

In [8]:
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

print("Running inference on test set...")
results = []
with torch.no_grad():
    for lr_imgs, filenames in test_loader:
        lr_imgs = lr_imgs.to(device)
        sr_imgs = model(lr_imgs)
        
        for i in range(sr_imgs.size(0)):
            img_tensor = sr_imgs[i].cpu().clamp(0, 1)
            img_pil = TF.to_pil_image(img_tensor).convert('L')
            image_array = np.array(img_pil).flatten()[::8]
            file_id = filenames[i].split('.')[0].replace('test_', 'gt_')
            
            row = [file_id] + image_array.tolist()
            results.append(row)

print("Inference completed!")
num_pixels = len(results[0]) - 1 if results else 0
columns = ['ID'] + [f'pixel_{j}' for j in range(num_pixels)]
df = pd.DataFrame(results, columns=columns)

df.to_csv('submission.csv', index=False)
print(f"Successfully saved to submission.csv with shape {df.shape}")

Running inference on test set...
Inference completed!
Successfully saved to submission.csv with shape (60, 81921)


In [10]:
!pip install -q huggingface_hub
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    
    api = HfApi()
    repo_id = "anuiitm/sup-res"
    
    print(f"Creating/checking repository {repo_id}...")
    api.create_repo(repo_id=repo_id, exist_ok=True)
    
    print("Uploading best_model.pth...")
    api.upload_file(
        path_or_fileobj="best_model.pth",
        path_in_repo="best_model.pth",
        repo_id=repo_id
    )
    print(f"\n✅ Model successfully uploaded to https://huggingface.co/{repo_id}")
except Exception as e:
    print(f"Error uploading to Hugging Face: {e}")
    print("\nMake sure you have:")
    print("1. Added your Hugging Face WRITE token as a Kaggle Secret named 'HF_TOKEN'")
    print("2. Attached the secrets to this notebook (Add-ons -> Secrets)")

Creating/checking repository anuiitm/sup-res...
Uploading best_model.pth...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


✅ Model successfully uploaded to https://huggingface.co/anuiitm/sup-res
